# Formal Spec Pipeline with BERTScore Judge

**Architecture:**
- **LLM 1** — Formal Spec Generator (3-step chain-of-thought: Annotation → Lifting → Translation)
- **LLM 2** — Summariser (formal specs → human-readable draft summary)
- **BERTScore Judge** — Replaces LLM 3. Uses two CrossEncoder models to score each operation:
  - `cross-encoder/stsb-roberta-large` for semantic similarity (STS)
  - `cross-encoder/nli-deberta-v3-base` for logical/negation consistency (NLI)
- **Refinement Loop** — BERTScore failures feed back into LLM 1 until threshold is met or max iterations reached.

## 0. Install & Import

In [1]:
!pip install --upgrade groq -q
!pip install sentence-transformers -q
!pip install bert-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00


In [2]:
import os, json, time, re
from typing import List, Dict, Any, Optional
from collections import defaultdict

import pandas as pd
from groq import Groq
from sentence_transformers import CrossEncoder

# ── CONFIGURATION ──────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()
GROQ_API_KEY = client.get_secret("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY in Kaggle Secrets.")
# Groq model fallback chain
MODEL_CHAIN          = ['llama-3.3-70b-versatile', 'llama-3.1-8b-instant']

# ── PIPELINE SETTINGS ──────────────────────────────────────────────
SIMILARITY_THRESHOLD = 0.9   # BERTScore combined_score threshold to PASS
MAX_ITERATIONS       = 5
BATCH_SIZE           = 3      # LLM batch size — keeps under Groq rate limits
INTER_BATCH_SLEEP    = 8      # seconds between LLM batches

# ── FILE PATHS ─────────────────────────────────────────────────────
# Place your ground-truth summary.json in /kaggle/input/<your-dataset>/
# and update DATASET_FOLDER below.
DATASET_FOLDER     = "/datasets/chandrimanandi/summary-actual"   # <-- CHANGE THIS
GROUND_TRUTH_PATH  = f"/kaggle/input/{DATASET_FOLDER}/summary.json"
FORMAL_SPECS_PATH  = '/kaggle/working/formal_specifications.json'
DRAFT_SUMMARY_PATH = '/kaggle/working/draft_summary.json'
JUDGE_REPORT_PATH  = '/kaggle/working/judge_report.json'
ITER_LOG_PATH      = '/kaggle/working/iteration_log.json'
OUTPUT_DIR         = '/kaggle/working/'

# ── BERTScore JUDGE SETTINGS ───────────────────────────────────────
STS_MODEL   = "cross-encoder/stsb-roberta-large"
NLI_MODEL   = "cross-encoder/nli-deberta-v3-base"
NLI_LABELS  = ["contradiction", "entailment", "neutral"]
STS_WEIGHT  = 0.50
NLI_WEIGHT  = 0.50

# ── INIT GROQ CLIENT ───────────────────────────────────────────────
client           = Groq(api_key=GROQ_API_KEY)
active_model_idx = 0

def current_model() -> str:
    return MODEL_CHAIN[active_model_idx]

print(f'Provider : Groq')
print(f'Models   : {MODEL_CHAIN}')
print(f'Threshold: {SIMILARITY_THRESHOLD}  Batch: {BATCH_SIZE}  Sleep: {INTER_BATCH_SLEEP}s')
print(f'Judge    : STS({STS_MODEL}) + NLI({NLI_MODEL})')

Provider : Groq
Models   : ['llama-3.3-70b-versatile', 'llama-3.1-8b-instant']
Threshold: 0.9  Batch: 3  Sleep: 8s
Judge    : STS(cross-encoder/stsb-roberta-large) + NLI(cross-encoder/nli-deberta-v3-base)


## 1. Helper Utilities

In [3]:
DAILY_QUOTA_EXHAUSTED = False

def _is_rpm_error(msg: str) -> bool:
    low = msg.lower()
    return '429' in msg and ('rate_limit' in low or 'rate limit' in low or 'too many requests' in low)

def _is_hard_quota_error(msg: str) -> bool:
    low = msg.lower()
    return '429' in msg and 'exceeded' in low and 'rate_limit' not in low

def _retry_wait(msg: str, default: int = 35) -> int:
    m = re.search(r'try again in ([\d.]+)s', msg)
    if m:
        return int(float(m.group(1))) + 2
    m = re.search(r'retry.after.*?(\d+)', msg.lower())
    if m:
        return int(m.group(1)) + 2
    return default


def call_llm(prompt: str,
             few_shot: Optional[List[Dict]] = None,
             max_retries: int = 3) -> str:
    """
    Groq-backed LLM caller (OpenAI-compatible chat format).
    RPM errors do NOT burn a retry attempt — just sleep and retry same model.
    Returns response text, or '' on failure.
    """
    global active_model_idx, DAILY_QUOTA_EXHAUSTED
    if DAILY_QUOTA_EXHAUSTED:
        print('   🚫 All models exhausted.')
        return ''

    def build_messages(prompt_text):
        messages = []
        if few_shot:
            for turn in few_shot:
                messages.append({
                    'role': turn['role'],
                    'content': turn['parts'][0]['text']
                })
        messages.append({'role': 'user', 'content': prompt_text})
        return messages

    attempt = 0
    while attempt < max_retries:
        attempt += 1
        model = current_model()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=build_messages(prompt),
                temperature=0.2,
                max_tokens=8192,
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            msg = str(e)
            print(f'   ⚠️  [{model}] attempt {attempt}: {msg[:180]}')

            if _is_rpm_error(msg):
                wait = _retry_wait(msg)
                print(f'   ⏱️  RPM limit → sleeping {wait}s...')
                time.sleep(wait)
                attempt -= 1  # RPM waits don't count as failed attempts
            elif _is_hard_quota_error(msg):
                next_idx = active_model_idx + 1
                if next_idx < len(MODEL_CHAIN):
                    print(f'   🔄 Quota on {model} → switching to {MODEL_CHAIN[next_idx]}')
                    active_model_idx = next_idx
                    attempt = 0
                else:
                    DAILY_QUOTA_EXHAUSTED = True
                    print('   🚫 All models exhausted.')
                    return ''
            elif attempt < max_retries:
                print(f'      retrying in 5s...')
                time.sleep(5)
            else:
                print(f'   ❌ Fatal after {max_retries} attempts.')
    return ''


def parse_json(raw: str) -> Any:
    """Strip markdown code fences then parse JSON."""
    text = re.sub(r'^```[a-z]*\n?', '', raw.strip())
    text = re.sub(r'\n?```$', '', text.strip()).strip()
    return json.loads(text)

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f'   💾 Saved → {path}')

def load_json(path):
    assert os.path.exists(path), f"File not found: {path}"
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def batched(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

print('✅ Helpers ready.')

✅ Helpers ready.


In [4]:
print('Available Groq models:')
try:
    for m in client.models.list().data:
        print(f'  {m.id}')
except Exception as e:
    print(f'Could not list models: {e}')

print('\nSmoke test...')
test = call_llm('Reply with just the word: OK')
print(f'Response : "{test}"')
print(f'Model    : {current_model()}')

Available Groq models:
  openai/gpt-oss-120b
  moonshotai/kimi-k2-instruct-0905
  openai/gpt-oss-20b
  llama-3.1-8b-instant
  allam-2-7b
  canopylabs/orpheus-v1-english
  meta-llama/llama-4-scout-17b-16e-instruct
  groq/compound-mini
  whisper-large-v3
  openai/gpt-oss-safeguard-20b
  groq/compound
  moonshotai/kimi-k2-instruct
  meta-llama/llama-prompt-guard-2-86m
  meta-llama/llama-prompt-guard-2-22m
  qwen/qwen3-32b
  canopylabs/orpheus-arabic-saudi
  llama-3.3-70b-versatile
  whisper-large-v3-turbo

Smoke test...
Response : "OK"
Model    : llama-3.3-70b-versatile


## 2. Load Ground Truth

In [5]:
ground_truth = load_json(GROUND_TRUTH_PATH)
all_operations: List[Dict] = []
for sec in ground_truth['sections']:
    for op in sec['operations']:
        all_operations.append({
            'sectionId': sec['sectionId'],
            'sectionTitle': sec['sectionTitle'],
            **op
        })

print(f'Ground truth: {len(ground_truth["sections"])} sections, {len(all_operations)} operations')
for s in ground_truth['sections']:
    print(f'  [{s["sectionId"]}] {s["sectionTitle"]} — {len(s["operations"])} ops')

Ground truth: 4 sections, 21 operations
  [1] Authentication — 3 ops
  [2] Plumber Onboarding Wizard (Steps 1–6) — 1 ops
  [3] Plumber Qualification — 7 ops
  [4] Complaint Lifecycle — 10 ops


## 3. LLM 1 — Formal Spec Generator

3-step chain-of-thought:
- **Step 1 — Annotation**: Extract actionable requirement sentences.
- **Step 2 — Lifting**: Convert sentences to Lifted-NL temporal/logical statements.
- **Step 3 — Translation**: Translate Lifted-NL to formal specs (Precondition, API call, Postcondition).

In [6]:
# ── STEP 1: ANNOTATION ────────────────────────────────────────────

ANNOT_SCHEMA = json.dumps({
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'operationId': {'type': 'string'},
            'operationName': {'type': 'string'},
            'actionable_sentences': {
                'type': 'array', 'items': {'type': 'string'},
                'description': 'Sentences encoding state transitions, guards, or system actions.'
            }
        },
        'required': ['operationId', 'operationName', 'actionable_sentences']
    }
}, indent=2)

FS_ANNOT = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'name': 'Signup',
        'purpose': 'Register a new account.',
        'precondition': 'The username u must not already exist in R.',
        'postcondition': 'R now contains an entry mapping u to hashed password p.'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'actionable_sentences': [
            'The username u must not already exist in R.',
            'R now contains an entry mapping u to hashed password p.'
        ]
    }])}]}
]

def step1_annotate(operations: List[Dict], extra: str = '') -> List[Dict]:
    print('\n🔹 STEP 1: Annotation')
    sys_p = f"""You are an expert in software engineering and formal methods.
Extract ONLY sentences that are actionable system requirements:
  • State transitions  ("state becomes X", "advances to Y", "reverts to Z")
  • Guards/Preconditions ("must be in state X", "must not already exist", "must match")
  • Postconditions ("is removed", "is created", "field is set to", "added to map")
Ignore introductory or descriptive sentences.
{extra}
Output ONLY a raw JSON array (no markdown, no preamble) matching this schema:
{ANNOT_SCHEMA}"""

    results = []
    bs = list(batched(operations, BATCH_SIZE))
    for i, batch in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        print(f'   Batch {i+1}/{len(bs)}: {len(batch)} ops')
        inp = [{'operationId': op['operationId'], 'name': op['name'],
                'purpose': op['purpose'], 'precondition': op['precondition'],
                'postcondition': op['postcondition']} for op in batch]
        raw = call_llm(sys_p + '\nJSON input:\n' + json.dumps(inp, indent=2), few_shot=FS_ANNOT)
        if not raw: continue
        try:
            r = parse_json(raw)
            if not isinstance(r, list): r = [r]
            results.extend(r)
            print(f'   ✅ {sum(len(x.get("actionable_sentences",[])) for x in r)} sentences')
        except Exception as e:
            print(f'   ❌ Parse: {e} | raw[:150]: {raw[:150]}')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)
    print(f'   Total annotated ops: {len(results)}')
    return results

print('✅ Step 1 defined.')

✅ Step 1 defined.


In [7]:
# ── STEP 2: LIFTING ───────────────────────────────────────────────

LIFT_SCHEMA = json.dumps({
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'operationId': {'type': 'string'},
            'operationName': {'type': 'string'},
            'source_sentence': {'type': 'string'},
            'lifted_nl': {
                'type': 'string',
                'description': "Structured logical statement: 'always','whenever','implies','becomes','dom()',map notation."
            }
        },
        'required': ['operationId', 'operationName', 'source_sentence', 'lifted_nl']
    }
}, indent=2)

LIFT_SYS = f"""You are an expert in formal methods and temporal logic.
Convert each actionable sentence to a Lifted Natural Language (Lifted NL) statement.
Rules:
  • Modal vocabulary: 'always','whenever','implies','must hold that','iff'
  • State changes: 'X becomes Y', 'X is added to map M', 'X is removed from M'
  • Named state vars: T (session token map), Qual (qualification map), Compl (complaint map),
    C/P/M (credential maps), QualState, ComplaintRecord
  • Actors: customer c, plumber p, manager m
  • One logical sentence per item.
Output ONLY a raw JSON array (no markdown, no preamble) matching:
{LIFT_SCHEMA}"""

FS_LIFT = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'The username u must not already exist in R.'
    }, {
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'R now contains an entry mapping u to hashed password p.'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'The username u must not already exist in R.',
        'lifted_nl': 'whenever signup(u,p,role) is called, it must hold that u \u2209 dom(R) where R=roleMap(role)'
    }, {
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'R now contains an entry mapping u to hashed password p.',
        'lifted_nl': 'always after signup(u,p,role) succeeds, R becomes R \u222a {u \u21a6 hash(p)}'
    }])}]}
]

def step2_lift(annotated: List[Dict]) -> List[Dict]:
    print('\n🔹 STEP 2: Lifting')
    flat = [{'operationId': x['operationId'], 'operationName': x['operationName'],
             'source_sentence': s}
            for x in annotated for s in x.get('actionable_sentences', [])]
    print(f'   Sentences to lift: {len(flat)}')
    results = []
    bs = list(batched(flat, BATCH_SIZE))
    for i, batch in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        print(f'   Batch {i+1}/{len(bs)}: {len(batch)} sentences')
        raw = call_llm(LIFT_SYS + '\nSentences:\n' + json.dumps(batch, indent=2), few_shot=FS_LIFT)
        if not raw: continue
        try:
            r = parse_json(raw)
            if not isinstance(r, list): r = [r]
            results.extend(r)
            print(f'   ✅ {len(r)} lifted')
        except Exception as e:
            print(f'   ❌ Parse: {e} | raw[:150]: {raw[:150]}')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)
    print(f'   Total lifted: {len(results)}')
    return results

print('✅ Step 2 defined.')

✅ Step 2 defined.


In [8]:
# ── STEP 3: TRANSLATION ───────────────────────────────────────────

TRANS_SCHEMA = json.dumps({
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'operationId':   {'type': 'string'},
            'operationName': {'type': 'string'},
            'LABEL': {'type': 'string', 'description': 'ALL_CAPS_SNAKE_CASE e.g. SIGNUP_OK'},
            'lifted_nl': {'type': 'string'},
            'Precondition': {
                'type': 'string',
                'description': 'Boolean predicate. Uses set/predicate logic symbols. No prose.'
            },
            'Function': {
                'type': 'string',
                'description': 'name(param:Type) -> ReturnType [HTTP_CODE]'
            },
            'Postcondition': {
                'type': 'string',
                'description': "New state via primed vars (R',T',Qual',Compl'). List ALL maps."
            }
        },
        'required': ['operationId','operationName','LABEL','lifted_nl',
                     'Precondition','Function','Postcondition']
    }
}, indent=2)

TRANS_SYS = f"""You are an expert Formal Specification Translator (predicate logic + set theory).

GLOBAL STATE:
  C,P,M : Map<username,hashedPassword>  (Customer/Plumber/Manager credentials)
  T     : Map<token,username>           (active session tokens)
  Qual  : Map<plumberId,QualState>
    QualState in {{STEP_1..STEP_6, ONBOARD_DONE, Q_INIT, UNDER_EXAM, TRAINING, ACTIVE, SUSPENDED, REJECTED}}
  Compl : Map<complaintId,ComplaintRecord>
    ComplaintRecord.state in {{RAISED,ASSIGNED,UNDER_EXAM_C,QUOTATION_RAISED,UNDER_EXEC,PAYMENT_PENDING,DONE,ABANDONED}}

RULES:
  1. LABEL        : ALL_CAPS_SNAKE_CASE unique per operation (e.g. LOGIN_OK)
  2. Precondition : pure Boolean predicate, no prose. Use set/predicate logic notation.
  3. Function     : name(param1:Type1, ...) -> ReturnType [HTTP_CODE]
  4. Postcondition: use primed vars, list ALL four maps even if unchanged
     e.g. T'=T union {{tok|->u}} AND C'=C AND Qual'=Qual AND Compl'=Compl
  5. Merge ALL lifted_nl items for the SAME operationId into ONE output record.

Output ONLY a raw JSON array (no markdown, no preamble) matching:
{TRANS_SCHEMA}"""

FS_TRANS = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'lifted_nl': 'whenever signup(u,p,role) called, u not in dom(R); after success R becomes R union {u|->hash(p)}'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'LABEL': 'SIGNUP_OK',
        'lifted_nl': 'whenever signup(u,p,role) called, u not in dom(R); after success R becomes R union {u|->hash(p)}',
        'Precondition': 'role in {Customer,Plumber,Manager} AND R=roleMap(role) AND u not in dom(R)',
        'Function': 'signup(u:Username, p:Password, role:Role) -> Unit [201 Created]',
        'Postcondition': "R'=R union {u|->hash(p)} AND T'=T AND Qual'=Qual AND Compl'=Compl"
    }])}]}
]

def step3_translate(lifted: List[Dict]) -> List[Dict]:
    print('\n🔹 STEP 3: Translation -> Formal Specs')
    grouped = defaultdict(list)
    for x in lifted: grouped[x['operationId']].append(x)
    op_groups = list(grouped.values())
    print(f'   Operations to formalise: {len(op_groups)}')
    results = []
    bs = list(batched(op_groups, BATCH_SIZE))
    for i, batch_ops in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        items = [x for ops in batch_ops for x in ops]
        print(f'   Batch {i+1}/{len(bs)}: {len(batch_ops)} ops ({len(items)} items)')
        raw = call_llm(TRANS_SYS + '\nLifted-NL:\n' + json.dumps(items, indent=2), few_shot=FS_TRANS)
        if not raw: continue
        try:
            r = parse_json(raw)
            if not isinstance(r, list): r = [r]
            results.extend(r)
            print(f'   ✅ {len(r)} specs')
        except Exception as e:
            print(f'   ❌ Parse: {e} | raw[:150]: {raw[:150]}')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)
    print(f'   Total formal specs: {len(results)}')
    return results

print('✅ Step 3 defined.')

✅ Step 3 defined.


In [9]:
print('='*60)
print('LLM 1 — FORMAL SPEC GENERATOR (initial run)')
print('='*60)

annotated_ops = step1_annotate(all_operations)
lifted_items  = step2_lift(annotated_ops)
formal_specs  = step3_translate(lifted_items)

save_json(formal_specs, FORMAL_SPECS_PATH)

if DAILY_QUOTA_EXHAUSTED:
    print('\n🚫 Halted — quota exhausted on all models.')
else:
    print(f'\n✅ LLM 1 done. {len(formal_specs)} specs -> {FORMAL_SPECS_PATH}')

LLM 1 — FORMAL SPEC GENERATOR (initial run)

🔹 STEP 1: Annotation
   Batch 1/7: 3 ops
   ✅ 6 sentences
   Batch 2/7: 3 ops
   ✅ 7 sentences
   Batch 3/7: 3 ops
   ✅ 6 sentences
   Batch 4/7: 3 ops
   ✅ 6 sentences
   Batch 5/7: 3 ops
   ✅ 7 sentences
   Batch 6/7: 3 ops
   ✅ 6 sentences
   Batch 7/7: 3 ops
   ✅ 6 sentences
   Total annotated ops: 21

🔹 STEP 2: Lifting
   Sentences to lift: 44
   Batch 1/15: 3 sentences
   ✅ 3 lifted
   Batch 2/15: 3 sentences
   ✅ 3 lifted
   Batch 3/15: 3 sentences
   ✅ 3 lifted
   Batch 4/15: 3 sentences
   ✅ 3 lifted
   Batch 5/15: 3 sentences
   ✅ 3 lifted
   Batch 6/15: 3 sentences
   ✅ 3 lifted
   Batch 7/15: 3 sentences
   ✅ 3 lifted
   Batch 8/15: 3 sentences
   ❌ Parse: Invalid \escape: line 12 column 84 (char 482) | raw[:150]: [
  {
    "operationId": "3-C3",
    "operationName": "Delete Plumber Account",
    "source_sentence": "The plumber must exist in the qualification ma
   Batch 9/15: 3 sentences
   ✅ 3 lifted
   Batch 10/15: 3 sentences

## 4. LLM 2 — Summariser

Converts formal specs back into a `draft_summary.json` that mirrors the schema of `summary.json`.

In [10]:
SUM_PROMPT = """You are an expert technical writer.
Given formal specs, reconstruct a human-readable summary JSON in EXACTLY this schema:
{
  "sections": [
    {
      "sectionId": <int>,
      "sectionTitle": <string>,
      "operations": [
        {
          "operationId": <string — MUST match input>,
          "name": <string>,
          "purpose": <string — 1-2 plain-English sentences>,
          "stateVariables": [<string>, ...],
          "precondition": <string — plain English, no math>,
          "postcondition": <string — plain English, no math>
        }
      ]
    }
  ]
}
Section grouping:
  1 — Authentication        (1-A, 1-B, 1-C)
  2 — Plumber Onboarding    (2-k)
  3 — Plumber Qualification (3-A..3-C3)
  4 — Complaint Lifecycle   (4-A..4-F)
Include EVERY operationId. Output ONLY raw JSON — no markdown, no preamble.
"""

def run_llm2(specs: List[Dict]) -> Dict:
    print('\n' + '='*60 + '\nLLM 2 — SUMMARISER\n' + '='*60)
    if DAILY_QUOTA_EXHAUSTED: return {'sections': []}
    raw = call_llm(SUM_PROMPT + '\nSpecs:\n' + json.dumps(specs, indent=2))
    if not raw: return {'sections': []}
    try:
        d = parse_json(raw)
        n = sum(len(s.get('operations', [])) for s in d.get('sections', []))
        print(f'   ✅ {len(d.get("sections",[]))} sections, {n} ops')
        return d
    except Exception as e:
        print(f'   ❌ Parse: {e}')
        return {'sections': []}

print('✅ LLM 2 defined.')

✅ LLM 2 defined.


## 5. BERTScore Judge (replaces LLM 3)

Uses two CrossEncoder models instead of an LLM judge:
- **STS** (`cross-encoder/stsb-roberta-large`): semantic paraphrase similarity (0–1)
- **NLI** (`cross-encoder/nli-deberta-v3-base`): logical consistency — catches negation flips
- **Combined score** = `STS_WEIGHT * sts_score + NLI_WEIGHT * nli_score`
- Operations with `combined_score < SIMILARITY_THRESHOLD` are flagged as **FAIL**

In [11]:
print(f'Loading STS model : {STS_MODEL}')
sts_model = CrossEncoder(STS_MODEL)

print(f'Loading NLI model : {NLI_MODEL}')
nli_model = CrossEncoder(NLI_MODEL, num_labels=3)

print('BERTScore judge models ready.')

Loading STS model : cross-encoder/stsb-roberta-large


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

BERTScore judge models ready.


In [12]:
def extract_op_texts(data: Dict) -> Dict[str, str]:
    """Extract {operationId: combined_text} for every operation in a summary JSON."""
    ops = {}
    for section in data.get('sections', []):
        for op in section.get('operations', []):
            op_id = op['operationId']
            text  = ' '.join(filter(None, [
                op.get('name', ''),
                op.get('purpose', ''),
                op.get('precondition', ''),
                op.get('postcondition', ''),
            ]))
            ops[op_id] = text.strip()
    return ops


def run_bertscore_judge(ground_truth: Dict, draft: Dict) -> Dict:
    """
    Scores draft vs ground truth using STS + NLI CrossEncoders.
    Returns a judge report compatible with the refinement loop.
    """
    print('\n' + '='*60 + '\nBERTSCORE JUDGE\n' + '='*60)

    gt_ops    = extract_op_texts(ground_truth)
    draft_ops = extract_op_texts(draft)

    common_ids    = sorted(set(gt_ops.keys()) & set(draft_ops.keys()))
    only_in_gt    = set(gt_ops.keys()) - set(draft_ops.keys())
    only_in_draft = set(draft_ops.keys()) - set(gt_ops.keys())

    print(f'   Ground truth ops : {len(gt_ops)}')
    print(f'   Draft ops        : {len(draft_ops)}')
    print(f'   Common ops       : {len(common_ids)}')
    if only_in_gt:    print(f'   Missing from draft: {sorted(only_in_gt)}')
    if only_in_draft: print(f'   Extra in draft    : {sorted(only_in_draft)}')

    if not common_ids:
        return {
            'overall_score': 0.0,
            'operation_scores': [],
            'missing_operations': sorted(only_in_gt),
            'overall_feedback': 'No common operations found between draft and ground truth.'
        }

    pairs = [(gt_ops[i], draft_ops[i]) for i in common_ids]

    # ── STS scores ──────────────────────────────────────────────────
    print('   Computing STS scores...')
    raw_sts   = sts_model.predict(pairs)
    sts_scores = [max(0.0, min(1.0, float(s))) for s in raw_sts]

    # ── NLI scores ──────────────────────────────────────────────────
    print('   Computing NLI scores...')
    nli_logits = nli_model.predict(pairs, apply_softmax=True)

    nli_scores = []
    nli_labels = []
    contradiction_p = []
    entailment_p    = []
    neutral_p       = []

    for logit in nli_logits:
        c, e, n = float(logit[0]), float(logit[1]), float(logit[2])
        nli_sim = e + 0.5 * n   # entailment=1, neutral=0.5, contradiction=0
        nli_scores.append(round(nli_sim, 4))
        nli_labels.append(NLI_LABELS[int(logit.argmax())])
        contradiction_p.append(round(c, 4))
        entailment_p.append(round(e, 4))
        neutral_p.append(round(n, 4))

    # ── Combined scores ─────────────────────────────────────────────
    combined_scores = [
        round(STS_WEIGHT * s + NLI_WEIGHT * n, 4)
        for s, n in zip(sts_scores, nli_scores)
    ]

    # ── Build per-operation report ───────────────────────────────────
    operation_scores = []
    for idx, op_id in enumerate(common_ids):
        score = combined_scores[idx]
        status = 'PASS' if score >= SIMILARITY_THRESHOLD else 'FAIL'
        label  = nli_labels[idx]

        # Generate human-readable feedback for the LLM 1 refinement prompt
        feedback_parts = []
        if label == 'contradiction':
            feedback_parts.append(f'CONTRADICTION detected (contradiction_p={contradiction_p[idx]:.3f}): semantic meaning is inverted — check negations, state transitions, and preconditions carefully')
        if sts_scores[idx] < 0.75:
            feedback_parts.append(f'low semantic similarity (sts={sts_scores[idx]:.3f}): the draft text diverges significantly from the ground truth')
        if nli_scores[idx] < 0.75:
            feedback_parts.append(f'low logical consistency (nli={nli_scores[idx]:.3f}): the logical relationship between pre/postconditions may be incorrect')
        feedback = '; '.join(feedback_parts) if feedback_parts else f'scores within acceptable range (sts={sts_scores[idx]:.3f}, nli={nli_scores[idx]:.3f})'

        operation_scores.append({
            'operationId':       op_id,
            'score':             score,
            'status':            status,
            'sts_score':         round(sts_scores[idx], 4),
            'nli_score':         nli_scores[idx],
            'nli_label':         label,
            'entailment_p':      entailment_p[idx],
            'neutral_p':         neutral_p[idx],
            'contradiction_p':   contradiction_p[idx],
            'feedback':          feedback,
            'ground_truth_text': gt_ops[op_id],
            'draft_text':        draft_ops[op_id],
        })

    # ── Summary stats ───────────────────────────────────────────────
    passed    = [op for op in operation_scores if op['status'] == 'PASS']
    failed    = [op for op in operation_scores if op['status'] == 'FAIL']
    contradictions = [op for op in operation_scores if op['nli_label'] == 'contradiction']
    overall_score  = round(sum(combined_scores) / len(combined_scores), 4)

    # Build overall feedback text for refinement
    fb_parts = []
    if contradictions:
        fb_parts.append(f'{len(contradictions)} contradiction(s) detected: {[op["operationId"] for op in contradictions]}')
    if failed:
        fb_parts.append(f'{len(failed)} op(s) below threshold ({SIMILARITY_THRESHOLD}): {[op["operationId"] for op in failed]}')
    overall_feedback = '. '.join(fb_parts) if fb_parts else 'All operations passed BERTScore threshold.'

    # ── Console output ──────────────────────────────────────────────
    print(f'\n   {"="*62}')
    print(f'   BERTSCORE JUDGE RESULTS  |  threshold = {SIMILARITY_THRESHOLD}')
    print(f'   {"="*62}')

    df = pd.DataFrame(operation_scores)
    display_cols = ['operationId', 'status', 'score', 'sts_score', 'nli_score', 'nli_label']
    print(df[display_cols].to_string(index=False))

    print(f'\n   PASSED: {len(passed)}/{len(operation_scores)}')
    print(f'   FAILED: {len(failed)}/{len(operation_scores)}')
    if failed:
        print(f'   ⚠️  Failed op IDs: {[op["operationId"] for op in failed]}')
    if contradictions:
        print(f'\n   🔴 CONTRADICTION detected in {len(contradictions)} op(s):')
        for op in contradictions:
            print(f'      {op["operationId"]}: contradiction_p={op["contradiction_p"]:.3f}  sts={op["sts_score"]:.3f}')
            print(f'        GT   : {op["ground_truth_text"][:120]}')
            print(f'        Draft: {op["draft_text"][:120]}')

    if only_in_gt:
        print(f'\n   ⚠️  Missing ops (in GT but not draft): {sorted(only_in_gt)}')

    print(f'\n   📊 Overall score: {overall_score:.4f}')

    return {
        'overall_score':       overall_score,
        'operation_scores':    operation_scores,
        'missing_operations':  sorted(only_in_gt),
        'overall_feedback':    overall_feedback,
        'judge_type':          'bertscore',
        'sts_model':           STS_MODEL,
        'nli_model':           NLI_MODEL,
        'threshold':           SIMILARITY_THRESHOLD,
    }

print('✅ BERTScore judge defined.')

✅ BERTScore judge defined.


## 6. Refinement Helper

Converts BERTScore failures into targeted feedback for LLM 1.

In [13]:
def build_refinement(report: Dict) -> str:
    """
    Converts a BERTScore judge report into a targeted refinement instruction
    string that is prepended to the LLM 1 annotation prompt.
    """
    missing  = report.get('missing_operations', [])
    feedback = report.get('overall_feedback', '')

    # Low-scoring: combined_score below threshold
    low_ops  = [
        op for op in report.get('operation_scores', [])
        if op.get('score', 1.0) < SIMILARITY_THRESHOLD
    ]

    # Also flag contradictions (even if combined_score barely passes)
    contradiction_ops = [
        op for op in report.get('operation_scores', [])
        if op.get('nli_label') == 'contradiction'
        and op['operationId'] not in {o['operationId'] for o in low_ops}
    ]

    all_priority = low_ops + contradiction_ops
    lines = '\n'.join(
        f"  [{op['operationId']}] score={op.get('score',0):.3f} "
        f"sts={op.get('sts_score',0):.3f} nli={op.get('nli_score',0):.3f} "
        f"nli_label={op.get('nli_label','?')}: {op.get('feedback','')}"
        for op in all_priority
    )
    focus = sorted(set([op['operationId'] for op in all_priority] + missing))

    return f"""
=== BERTSCORE REFINEMENT FEEDBACK (from previous iteration) ===
Overall: {feedback}

Operations needing improvement (BERTScore FAIL or CONTRADICTION):
{lines or '  (none specifically flagged)'}

Missing operations (present in ground truth but absent from draft): {missing or '(none)'}
Focus extra attention on operationIds: {focus}

Fix guidance:
  • CONTRADICTION means the pre/postcondition logic is semantically INVERTED —
    check for negation errors ("must exist" vs "must NOT exist") and state transition direction.
  • Low sts_score means the overall meaning has drifted — stay closer to the ground truth wording.
  • Low nli_score means the logical relationship is inconsistent.
  • Exact QualState names: STEP_1..STEP_6, ONBOARD_DONE, Q_INIT, UNDER_EXAM, TRAINING, ACTIVE, SUSPENDED, REJECTED
  • Exact Compl.state:     RAISED, ASSIGNED, UNDER_EXAM_C, QUOTATION_RAISED, UNDER_EXEC, PAYMENT_PENDING, DONE, ABANDONED
  • Postconditions MUST list ALL four maps (C'/P'/M', T', Qual', Compl') even if unchanged.
=== END REFINEMENT FEEDBACK ===
"""

print('✅ Refinement helper defined.')

✅ Refinement helper defined.


## 7. Main Iterative Loop

Runs: **LLM 2 → BERTScore Judge → (if FAIL) LLM 1 refinement → repeat**

Stops when `overall_score >= SIMILARITY_THRESHOLD` or `MAX_ITERATIONS` is reached.

In [ ]:
print('\n' + '='*60)
print(f'  ITERATIVE PIPELINE  threshold={SIMILARITY_THRESHOLD}  max_iter={MAX_ITERATIONS}')
print('='*60)

iter_log      = []
current_score = 0.0
current_specs = formal_specs

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f'\n--- ITERATION {iteration}/{MAX_ITERATIONS} ---')

    if DAILY_QUOTA_EXHAUSTED:
        print('🚫 Quota exhausted — stopping.')
        break

    # ── LLM 2: formal specs → draft summary ─────────────────────────
    draft = run_llm2(current_specs)
    save_json(draft, DRAFT_SUMMARY_PATH)

    # ── BERTScore Judge (replaces LLM 3) ────────────────────────────
    report = run_bertscore_judge(ground_truth, draft)
    current_score = report.get('overall_score', 0.0)
    report['iteration'] = iteration
    save_json(report, JUDGE_REPORT_PATH)

    iter_log.append({
        'iteration':    iteration,
        'score':        current_score,
        'passed':       sum(1 for op in report.get('operation_scores', []) if op['status'] == 'PASS'),
        'failed':       sum(1 for op in report.get('operation_scores', []) if op['status'] == 'FAIL'),
        'contradictions': sum(1 for op in report.get('operation_scores', []) if op['nli_label'] == 'contradiction'),
        'feedback':     report.get('overall_feedback', ''),
    })

    flag = '✅' if current_score >= SIMILARITY_THRESHOLD else '❌'
    print(f'\n  {flag} Score: {current_score:.4f}  (threshold: {SIMILARITY_THRESHOLD})')

    if current_score >= SIMILARITY_THRESHOLD:
        print('  🎉 THRESHOLD REACHED!')
        break

    if iteration == MAX_ITERATIONS:
        print('  ⚠️  Max iterations reached.')
        break

    # ── Refinement: feed BERTScore failures back into LLM 1 ─────────
    print(f'  🔁 Refining for iteration {iteration+1}...')
    extra = build_refinement(report)
    ann   = step1_annotate(all_operations, extra=extra)
    lift  = step2_lift(ann)
    spec  = step3_translate(lift)
    current_specs = spec
    save_json(current_specs, f'/kaggle/working/formal_specifications_iter{iteration+1}.json')
    time.sleep(3)

print('\n' + '='*60 + '\n  DONE\n' + '='*60)
for log in iter_log:
    f = '✅' if log['score'] >= SIMILARITY_THRESHOLD else '❌'
    print(f"  Iter {log['iteration']}: score={log['score']:.4f} {f}  "
          f"passed={log['passed']} failed={log['failed']} contradictions={log['contradictions']}")
save_json(iter_log, ITER_LOG_PATH)
print(f'\n  Final score     : {current_score:.4f}')
print(f'  Result          : {"PASSED ✅" if current_score >= SIMILARITY_THRESHOLD else "NOT REACHED ⚠️"}')


  ITERATIVE PIPELINE  threshold=0.9  max_iter=5

--- ITERATION 1/5 ---

LLM 2 — SUMMARISER
   ✅ 4 sections, 20 ops
   💾 Saved → /kaggle/working/draft_summary.json

BERTSCORE JUDGE
   Ground truth ops : 21
   Draft ops        : 20
   Common ops       : 20
   Missing from draft: ['3-C3']
   Computing STS scores...
   Computing NLI scores...

   BERTSCORE JUDGE RESULTS  |  threshold = 0.9
operationId status  score  sts_score  nli_score     nli_label
        1-A   FAIL 0.8903     0.7833     0.9973    entailment
        1-B   FAIL 0.8565     0.8199     0.8931    entailment
        1-C   FAIL 0.3910     0.7776     0.0045 contradiction
        2-k   FAIL 0.8881     0.7801     0.9962    entailment
        3-A   FAIL 0.8793     0.8381     0.9205    entailment
       3-B1   PASS 0.9125     0.8272     0.9978    entailment
       3-B2   PASS 0.9482     0.8993     0.9971    entailment
       3-B3   PASS 0.9303     0.8638     0.9968    entailment
       3-C1   PASS 0.9540     0.9114     0.9965    e

## 8. Inspect Results

In [ ]:
print('\n📋 SAMPLE FORMAL SPECS (first 3)')
for sp in current_specs[:3]:
    print(f"\n[{sp.get('LABEL','?')}] {sp.get('operationName','?')} ({sp.get('operationId','?')})")
    print(f"  Pre : {sp.get('Precondition','—')}")
    print(f"  Fn  : {sp.get('Function','—')}")
    print(f"  Post: {sp.get('Postcondition','—')}")

In [ ]:
draft = load_json(DRAFT_SUMMARY_PATH)
print('\n📋 DRAFT SUMMARY')
for s in draft.get('sections', []):
    ops = s.get('operations', [])
    print(f"  §{s.get('sectionId')}: {s.get('sectionTitle')} — {len(ops)} ops")
    for op in ops:
        print(f"    [{op.get('operationId')}] {op.get('name')}")

In [ ]:
report = load_json(JUDGE_REPORT_PATH)
print('\n📊 BERTSCORE JUDGE REPORT (final iteration)')
print(f"  Score      : {report.get('overall_score')}")
print(f"  Judge type : {report.get('judge_type')} (STS + NLI)")
print(f"  Feedback   : {report.get('overall_feedback', '')}")
print(f"  Missing    : {report.get('missing_operations', [])}")
print('\n  Per-operation:')
for op in report.get('operation_scores', []):
    icon = '✅' if op.get('status') == 'PASS' else ('🔴' if op.get('nli_label') == 'contradiction' else '⚠️ ')
    print(f"  {icon} [{op.get('operationId')}]  score={op.get('score',0):.4f}  "
          f"sts={op.get('sts_score',0):.4f}  nli={op.get('nli_score',0):.4f}  "
          f"label={op.get('nli_label','?')}")

In [ ]:
# Save full per-operation results to CSV
import pandas as pd

report = load_json(JUDGE_REPORT_PATH)
op_scores = report.get('operation_scores', [])

if op_scores:
    df = pd.DataFrame(op_scores)
    csv_path = os.path.join(OUTPUT_DIR, 'judge_results.csv')
    df.to_csv(csv_path, index=False)
    print(f'Full results saved → {csv_path}')
    display_cols = ['operationId', 'status', 'score', 'sts_score', 'nli_score', 'nli_label']
    print(df[display_cols].to_string(index=False))

# Save failed-ops JSON for manual inspection or retry
failed_ops = [op for op in op_scores if op.get('status') == 'FAIL']
if failed_ops:
    failed_ids = [op['operationId'] for op in failed_ops]
    failed_payload = {
        'similarity_threshold':  SIMILARITY_THRESHOLD,
        'failed_operation_ids':  failed_ids,
        'operations': [
            op
            for section in ground_truth['sections']
            for op in section['operations']
            if op['operationId'] in failed_ids
        ]
    }
    failed_path = os.path.join(OUTPUT_DIR, 'failed_ops.json')
    with open(failed_path, 'w') as f:
        json.dump(failed_payload, f, indent=2)
    print(f'Failed ops payload saved → {failed_path}')
else:
    print('✅ All operations passed. No failed ops to save.')